# Initialize

In [1]:
import os
import sys

import pandas as pd
import polars as pl

os.chdir("../../")
sys.path.insert(0, os.getcwd())

In [2]:
from morai.experience import charters, experience
from morai.utils import custom_logger, helpers

In [3]:
# default is "plotly_mimetype+notebook", however that takes up space.
# "plotly_mimetype+notebook_connected" seems to save space
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

## Normalize

Normalization first will calculate the relative risk for the feature/s that are being normalized. The relative risk is the percentage above the aggregate. The normalized value is then calculated as `current value / risk`.

For example:
  - If an year `2020` has a 10% higher rate than `2019` the `aggregate risk` is `0.905` for `2019` and `1.022` for `2020`.
  - The resulting rates should only differ by lob and the lob will be the same for a given year, if the rates are consistent by year for lob.
  - If wanting the rates to be consitent by year, then the relative_cols argument should be passed with lob

To normalize:
  - The rates are then divided by these risks to get a normalized rate by year.

In [4]:
experience_df = pd.read_csv("tests/files/experience/simple_normalization.csv")
experience_df = experience_df[
    ["year", "lob", "year_lob_multi_rate", "year_rate", "lob_rate", "yearlob_rate"]
]

In [5]:
experience_df["aggregate_risk"] = experience.calc_relative_risk(
    df=experience_df,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="aggregate",
)["relative_risk"]
experience_df["reference_risk"] = experience.calc_relative_risk(
    df=experience_df,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="reference",
)["relative_risk"]
experience_df["reference_by_risk"] = experience.calc_relative_risk(
    df=experience_df,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="reference",
    relative_cols=["lob"],
)["relative_risk"]
experience_df["subset_risk"] = experience.calc_relative_risk(
    df=experience_df,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="subset",
    relative_cols=["lob"],
    subset_dict={'year':[2019,2020]},
)["relative_risk"]
experience_df

,year,lob,year_lob_multi_rate,year_rate,lob_rate,yearlob_rate,aggregate_risk,reference_risk,reference_by_risk,subset_risk
0,2019,UL,1.1000,1.0,1.1,1.00,0.905953,1.000000,1.000,0.928074
1,2020,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926
2,2021,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926
3,2022,UL,1.3310,1.1,1.1,1.10,1.048749,1.157619,1.210,1.122970
4,2019,TERM,1.0000,1.0,1.0,1.00,0.905953,1.000000,1.000,0.952381
5,2020,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619
6,2021,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619
7,2022,TERM,1.1000,1.1,1.0,1.00,1.048749,1.157619,1.100,1.047619


In [6]:
experience.normalize(
    df=experience_df,
    features=["year"],
    normalize_col="year_lob_multi_rate",
    add_norm_col=True,
)

,year,lob,year_lob_multi_rate,year_rate,lob_rate,yearlob_rate,aggregate_risk,reference_risk,reference_by_risk,subset_risk,risk_numerator,baseline_ratio,year_lob_multi_rate_norm
0,2019,UL,1.1000,1.0,1.1,1.00,0.905953,1.000000,1.000,0.928074,2.1000,2.318,1.214190
1,2020,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926,2.3705,2.318,1.242362
2,2021,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926,2.3705,2.318,1.242362
3,2022,UL,1.3310,1.1,1.1,1.10,1.048749,1.157619,1.210,1.122970,2.4310,2.318,1.269131
4,2019,TERM,1.0000,1.0,1.0,1.00,0.905953,1.000000,1.000,0.952381,2.1000,2.318,1.103810
5,2020,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619,2.3705,2.318,1.075638
6,2021,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619,2.3705,2.318,1.075638
7,2022,TERM,1.1000,1.1,1.0,1.00,1.048749,1.157619,1.100,1.047619,2.4310,2.318,1.048869


## Polars

In [7]:
experience_df = pd.read_csv("tests/files/experience/simple_normalization.csv")
experience_df = experience_df[
    ["year", "lob", "year_lob_multi_rate", "year_rate", "lob_rate", "yearlob_rate"]
]

In [8]:
experience_pf = pl.from_pandas(experience_df).lazy()

In [9]:
experience_df["aggregate_risk"] = experience.calc_relative_risk(
    df=experience_pf,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="aggregate",
).collect().to_pandas()["relative_risk"]
experience_df["reference_risk"] = experience.calc_relative_risk(
    df=experience_pf,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="reference",
).collect().to_pandas()["relative_risk"]
experience_df["reference_by_risk"] = experience.calc_relative_risk(
    df=experience_pf,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="reference",
    relative_cols=["lob"],
).collect().to_pandas()["relative_risk"]
experience_df["subset_risk"] = experience.calc_relative_risk(
    df=experience_pf,
    features=["year"],
    risk_col="year_lob_multi_rate",
    relative_to="subset",
    relative_cols=["lob"],
    subset_dict={'year':[2019,2020]},
).collect().to_pandas()["relative_risk"]
experience_df

,year,lob,year_lob_multi_rate,year_rate,lob_rate,yearlob_rate,aggregate_risk,reference_risk,reference_by_risk,subset_risk
0,2019,UL,1.1000,1.0,1.1,1.00,0.905953,1.000000,1.000,0.928074
1,2020,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926
2,2021,UL,1.2705,1.1,1.1,1.05,1.022649,1.128810,1.155,1.071926
3,2022,UL,1.3310,1.1,1.1,1.10,1.048749,1.157619,1.210,1.122970
4,2019,TERM,1.0000,1.0,1.0,1.00,0.905953,1.000000,1.000,0.952381
5,2020,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619
6,2021,TERM,1.1000,1.1,1.0,1.00,1.022649,1.128810,1.100,1.047619
7,2022,TERM,1.1000,1.1,1.0,1.00,1.048749,1.157619,1.100,1.047619


In [10]:
experience.normalize(
    df=experience_pf,
    features=["year"],
    normalize_col="year_lob_multi_rate",
    add_norm_col=True,
).collect().to_pandas()

,year,lob,year_lob_multi_rate,year_rate,lob_rate,yearlob_rate,risk_numerator,baseline_ratio,year_lob_multi_rate_norm
0,2019,UL,1.1000,1.0,1.1,1.00,2.1000,2.318,1.214190
1,2020,UL,1.2705,1.1,1.1,1.05,2.3705,2.318,1.242362
2,2021,UL,1.2705,1.1,1.1,1.05,2.3705,2.318,1.242362
3,2022,UL,1.3310,1.1,1.1,1.10,2.4310,2.318,1.269131
4,2019,TERM,1.0000,1.0,1.0,1.00,2.1000,2.318,1.103810
5,2020,TERM,1.1000,1.1,1.0,1.00,2.3705,2.318,1.075638
6,2021,TERM,1.1000,1.1,1.0,1.00,2.3705,2.318,1.075638
7,2022,TERM,1.1000,1.1,1.0,1.00,2.4310,2.318,1.048869


## Reload

In [36]:
import importlib

importlib.reload(experience)

<module 'morai.experience.experience' from 'C:\\Users\\johnk\\Desktop\\github\\morai\\morai\\experience\\experience.py'>